# LeetCode 473 — Matchsticks to Square bằng Python

Link đề: https://leetcode.com/problems/matchsticks-to-square/ · Độ khó: **Medium** · Chủ đề: Array, Backtracking (quay lui + cắt tỉa), Bitmask, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def makesquare(self, matchsticks: List[int]) -> bool:
        ...
```

Ghi chú: bài này **không** phải Premium. Đây là bài "chia mảng thành 4 nhóm có tổng bằng nhau" nên rất hay gặp trong phỏng vấn dưới nhiều dạng (Partition to K Equal Sum Subsets, Fair Distribution of Cookies, ...).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: dùng **tất cả** que diêm, **mỗi que đúng một lần**, không bẻ gãy, ghép thành **một hình vuông** (4 cạnh bằng nhau); biết đổi bài toán thành "chia mảng thành 4 nhóm có tổng bằng nhau".
2. Nắm các công cụ nền: `divmod`, `sum`/`max`, sắp xếp giảm dần, đệ quy quay lui (backtracking), và **bitmask** (dùng một số nguyên làm tập hợp các que đã dùng).
3. Biết các **điều kiện cắt tỉa sớm**: tổng không chia hết cho 4, que dài hơn cạnh, ít hơn 4 que, và hai mẹo tăng tốc quay lui: **đặt que dài trước** và **bỏ qua cạnh trùng trạng thái**.
4. Cài đặt 4 cách khác nhau về ý tưởng: **vét cạn gán que vào 4 cạnh (`4^n`) → quay lui có cắt tỉa → DP bitmask (`2^n * n`) → quay lui theo từng cạnh (lấp đầy cạnh này rồi mới sang cạnh khác)**, và so sánh độ phức tạp.
5. Tránh các lỗi kinh điển: quên kiểm tra `tổng % 4`, quên so với que dài nhất, quay lui không sắp xếp giảm dần (chậm), nhận nhầm "tổng đúng" là "đủ", quên hoàn tác (`-=`) khi quay lui.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `matchsticks`, trong đó `matchsticks[i]` là **độ dài** của que diêm thứ `i`. Ta muốn dùng **tất cả** các que để xếp thành **một hình vuông**. **Không được bẻ gãy** que nào, nhưng được **nối** các que lại với nhau, và **mỗi que phải được dùng đúng một lần**.

Trả về `true` nếu xếp được hình vuông, ngược lại trả về `false`.

**Ví dụ 1:** `matchsticks = [1,1,2,2,2]` → `true`. Cạnh vuông dài `2`: hai que dài `2` làm hai cạnh, que `2` thứ ba làm cạnh thứ ba, hai que `1` nối lại làm cạnh thứ tư (`1 + 1 = 2`).

**Ví dụ 2:** `matchsticks = [3,3,3,3,4]` → `false`. Tổng `= 16`, mỗi cạnh phải dài `4`; nhưng bốn que `3` không ghép được thành `4` và que `4` đã đủ một cạnh, còn dư `3,3,3,3` không chia thành 3 cạnh `4`.

**Ràng buộc:**

- `1 <= matchsticks.length <= 15`
- `1 <= matchsticks[i] <= 10^8`

Con số `15` là gợi ý cực mạnh: `2^15 = 32768` rất nhỏ nên **bitmask** hoặc **quay lui** đều được; còn `4^15 ≈ 10^9` thì vét cạn thô sẽ quá chậm.

In [1]:
import itertools
import random
import time
from typing import List

examples = [([1, 1, 2, 2, 2], True), ([3, 3, 3, 3, 4], False)]
for ms, ans in examples:
    print(f"matchsticks = {ms}  tong = {sum(ms)}  canh = {sum(ms)/4}  -> {ans}")

matchsticks = [1, 1, 2, 2, 2]  tong = 8  canh = 2.0  -> True
matchsticks = [3, 3, 3, 3, 4]  tong = 16  canh = 4.0  -> False


## 3. Kiến thức nền

### 3.1. `divmod`, `sum`, `max`: kiểm tra tổng chia hết cho 4

`divmod(a, b)` trả về cặp `(a // b, a % b)` (thương và dư) trong một lần gọi. Ta dùng nó để tính **độ dài cạnh** và kiểm tra có chia hết cho 4 không.

In [2]:
print(divmod(16, 4))        # (4, 0)  chia hết, mỗi cạnh = 4
print(divmod(10, 4))        # (2, 2)  dư 2 -> chắc chắn không ghép được
ms = [1, 1, 2, 2, 2]
canh, du = divmod(sum(ms), 4)
print("tong =", sum(ms), "canh =", canh, "du =", du, "que dai nhat =", max(ms))

(4, 0)
(2, 2)
tong = 8 canh = 2 du = 0 que dai nhat = 2


### 3.2. Sắp xếp giảm dần và vì sao "que dài trước"

`sorted(a, reverse=True)` hoặc `a.sort(reverse=True)` sắp xếp từ lớn đến bé. Khi quay lui, ta đặt que **dài** trước vì que dài làm cạnh "đầy" rất nhanh, các nhánh sai bị loại sớm (ít nhánh phải duyệt hơn).

In [3]:
a = [1, 1, 2, 2, 2]
print(sorted(a, reverse=True))    # [2, 2, 2, 1, 1]
a.sort(reverse=True)               # sắp xếp tại chỗ
print(a)

[2, 2, 2, 1, 1]
[2, 2, 2, 1, 1]


### 3.3. Đệ quy quay lui (backtracking) và hoàn tác

Quay lui = thử một lựa chọn → đi sâu → nếu hỏng thì **hoàn tác** lựa chọn rồi thử lựa chọn khác. Với 4 cạnh `edges = [0, 0, 0, 0]`, "đặt que `u` vào cạnh `i`" là `edges[i] += que`, và hoàn tác là `edges[i] -= que`. Quên hoàn tác là lỗi phổ biến nhất.

In [4]:
edges = [0, 0, 0, 0]
que = 2
edges[1] += que          # thử đặt que 2 vào cạnh 1
print("sau khi đặt :", edges)
edges[1] -= que          # hoàn tác để thử cạnh khác
print("sau khi hoàn tác:", edges)

sau khi đặt : [0, 2, 0, 0]
sau khi hoàn tác: [0, 0, 0, 0]


### 3.4. Bitmask: một số nguyên làm tập hợp

Với `n <= 15` que, ta dùng **một số nguyên `mask`** trong đó bit `i` bằng `1` nghĩa là "que `i` đã dùng". Các thao tác cần biết:

- `mask >> i & 1` hoặc `(mask >> i) & 1`: que `i` đã dùng chưa? (`>>` ưu tiên cao hơn `&`.)
- `mask | (1 << i)`: thêm que `i` vào tập.
- `(1 << n) - 1`: mask "đã dùng hết `n` que" (toàn bit 1).

In [5]:
n = 5
mask = 0
mask = mask | (1 << 0)         # dùng que 0
mask = mask | (1 << 3)         # dùng que 3
print(bin(mask), "= 0b01001")
print("que 3 dung chua?", bool(mask >> 3 & 1), "| que 1 dung chua?", bool(mask >> 1 & 1))
print("mask day du khi n = 5:", (1 << n) - 1, bin((1 << n) - 1))

0b1001 = 0b01001
que 3 dung chua? True | que 1 dung chua? False
mask day du khi n = 5: 31 0b11111


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Bước 1: đổi đề bài.** Hình vuông có 4 cạnh bằng nhau và mỗi cạnh được ghép từ vài que. Vậy bài toán tương đương: **chia các que thành đúng 4 nhóm, mỗi nhóm có tổng bằng nhau** (và mỗi nhóm không rỗng, nhưng điều này tự thỏa vì mỗi cạnh dài `> 0`).

**Bước 2: các điều kiện cần (loại sớm).**

1. Tổng `S` phải chia hết cho 4, nếu không thì **không có** độ dài cạnh nguyên. Cạnh `x = S / 4`.
2. Que dài nhất phải `<= x`, vì không bẻ được que.
3. Phải có ít nhất 4 que (mỗi cạnh ít nhất một que). Điều kiện này thực ra đã nằm trong điều kiện 2: nếu `n < 4` thì que lớn nhất `>= S / n > S / 4 = x`, nên bị loại. Ta vẫn ghi `n < 4` riêng cho dễ hiểu.

**Tính tay ví dụ 1:** `[1,1,2,2,2]`: `S = 8`, `x = 2`, que lớn nhất `2 <= 2` ✓. Sắp giảm dần `[2,2,2,1,1]`. Đặt `2 → cạnh A`, `2 → cạnh B`, `2 → cạnh C`, rồi `1`, `1` cùng vào cạnh D → `1 + 1 = 2` ✓. Ba cạnh đầy ngay, cạnh D được `1 + 1`.

**Tính tay ví dụ 2:** `[3,3,3,3,4]`: `S = 16`, `x = 4`. Que `4` → cạnh A (đầy). Ba que `3` còn lại mỗi cái vào một cạnh B, C, D (`3`), còn que `3` cuối: cạnh nào cũng `3 + 3 = 6 > 4`. Không có chỗ → `false`.

**Câu hỏi then chốt:** làm sao thử các cách chia mà không phải duyệt `4^n` khả năng?

In [6]:
def dieu_kien_can(ms):
    """True nếu vượt qua các điều kiện cần (chưa chắc xếp được)."""
    canh, du = divmod(sum(ms), 4)
    if du != 0:
        return False
    if len(ms) < 4:
        return False
    return max(ms) <= canh

for ms in ([1, 1, 2, 2, 2], [3, 3, 3, 3, 4], [1, 2, 3], [5, 5, 5, 5, 4], [2, 2, 2, 2, 2, 2, 2, 2]):
    print(ms, "->", dieu_kien_can(ms))

[1, 1, 2, 2, 2] -> True
[3, 3, 3, 3, 4] -> True
[1, 2, 3] -> False
[5, 5, 5, 5, 4] -> True
[2, 2, 2, 2, 2, 2, 2, 2] -> True


## 5. Cách 1 — Brute Force / baseline: gán MỌI que vào MỘT trong 4 cạnh

**Ý tưởng:** mỗi que có 4 lựa chọn (cạnh `0..3`). Duyệt hết `4^n` cách gán; với mỗi cách tính tổng 4 cạnh; nếu cả bốn cạnh bằng nhau thì trả về `True`.

Với `n = 15` ta có `4^15 ≈ 1,07 × 10^9` cách gán, mỗi cách tính tổng tốn `O(n)`, quá chậm. Nhưng cách này **rất đúng và dễ**, nên ta dùng làm **oracle** để kiểm tra các cách khác trên input nhỏ (`n <= 8`).

In [7]:
def make_square_brute(ms):
    """Vét cạn: gán mỗi que vào cạnh 0..3, kiểm tra 4 cạnh bằng nhau."""
    n = len(ms)
    for gan in itertools.product(range(4), repeat=n):     # gan[i] = cạnh của que i
        tong = [0, 0, 0, 0]
        for i, c in enumerate(gan):
            tong[c] += ms[i]
        if tong[0] == tong[1] == tong[2] == tong[3]:
            return True
    return False

for ms, ans in examples:
    print(ms, "->", make_square_brute(ms), "(mong doi", ans, ")")

[1, 1, 2, 2, 2] -> True (mong doi True )
[3, 3, 3, 3, 4] -> False (mong doi False )


**Độ phức tạp:** Time `O(4^n * n)`, Space `O(n)`.

**Nút thắt:** hai điểm lãng phí lớn:

1. Ta vẫn đi tiếp những nhánh mà **một cạnh đã vượt `x`**, trong khi không thể cứu vãn.
2. Rất nhiều cách gán **đối xứng**: hoán đổi tên hai cạnh cho cùng một hình vuông, vậy ta duyệt cùng một lời giải nhiều lần (tối đa `4! = 24` lần).

Hãy đo thời gian để thấy `4^n` tăng nhanh thế nào.

In [8]:
for n in (4, 6, 8, 9):
    ms = [1] * (n - 1) + [2]          # tổng lẻ -> không xếp được, buộc duyệt hết (vét cạn không dừng sớm)
    t0 = time.perf_counter()
    kq = make_square_brute(ms)
    print(f"n = {n}: {4**n:>7} cach gan, kq = {kq}, mat {time.perf_counter() - t0:.3f}s")

n = 4:     256 cach gan, kq = False, mat 0.000s
n = 6:    4096 cach gan, kq = False, mat 0.002s
n = 8:   65536 cach gan, kq = False, mat 0.032s


n = 9:  262144 cach gan, kq = False, mat 0.140s


## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét làm quay lui trở nên nhanh:

**(a) Cắt tỉa theo độ dài.** Khi đặt que `u` vào cạnh `i`, nếu `edges[i] + que > x` thì **nhánh này chắc chắn hỏng**, bỏ ngay mà không đi sâu.

**(b) Bỏ qua cạnh đối xứng.** Nếu hai cạnh `i-1` và `i` đang có **cùng độ dài hiện tại** thì đặt que vào cạnh nào cũng cho kết quả giống nhau (chỉ khác tên cạnh). Vậy chỉ cần thử **một** trong hai. Cụ thể: nếu `edges[i-1] == edges[i]` thì bỏ qua cạnh `i` (xử lý đúng cả khi các cạnh còn là `0` ở giai đoạn đầu: cạnh trống cũng giống nhau).

**(c) Que dài trước.** Sắp xếp giảm dần để các que lớn (ít linh hoạt nhất) được đặt trước, nhánh sai lộ ra sớm.

Một hướng khác (đột phá về bộ nhớ): trạng thái "đã dùng que nào" là một **tập con** của `n` que, tức `2^n` trạng thái; với mỗi tập, **cạnh đang dựng dở** có độ dài `tổng(tập) % x`, vì các cạnh trước đã đầy chẵn `x`. Đó là **DP bitmask**.

## 7. Các cách tiếp theo

### Cách 2 — Quay lui đặt từng que vào 4 cạnh, có cắt tỉa

**Giải thích từng đoạn:**

1. `canh, du = divmod(sum(ms), 4)`; nếu `du != 0` hoặc `canh < max(ms)` thì trả về `False` ngay.
2. Sắp xếp que giảm dần (đặt que dài trước).
3. `dfs(u)`: đặt que thứ `u`. Nếu `u == n` thì mọi que đã được đặt mà không cạnh nào vượt `x`, tổng bằng `4x` nên cả bốn cạnh đều đúng bằng `x` → `True`.
4. Với mỗi cạnh `i` trong `0..3`: nếu `i > 0` và `edges[i-1] == edges[i]` thì `continue` (cạnh đối xứng); đặt que; nếu `edges[i] <= x` và `dfs(u+1)` thì trả `True`; hoàn tác.

In [9]:
def make_square_backtrack(ms):
    canh, du = divmod(sum(ms), 4)
    if du != 0 or canh < max(ms):
        return False
    a = sorted(ms, reverse=True)
    n = len(a)
    edges = [0, 0, 0, 0]

    def dfs(u):
        if u == n:
            return True
        for i in range(4):
            if i > 0 and edges[i - 1] == edges[i]:     # cạnh đối xứng với cạnh i-1: bỏ qua
                continue
            edges[i] += a[u]
            if edges[i] <= canh and dfs(u + 1):
                return True
            edges[i] -= a[u]                           # hoàn tác
        return False

    return dfs(0)

for ms, ans in examples:
    print(ms, "->", make_square_backtrack(ms), "(mong doi", ans, ")")

[1, 1, 2, 2, 2] -> True (mong doi True )
[3, 3, 3, 3, 4] -> False (mong doi False )


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def makesquare(self, matchsticks: List[int]) -> bool:
        canh, du = divmod(sum(matchsticks), 4)
        if du != 0 or canh < max(matchsticks):
            return False
        a = sorted(matchsticks, reverse=True)
        n = len(a)
        edges = [0, 0, 0, 0]

        def dfs(u):
            if u == n:
                return True
            for i in range(4):
                if i > 0 and edges[i - 1] == edges[i]:
                    continue
                edges[i] += a[u]
                if edges[i] <= canh and dfs(u + 1):
                    return True
                edges[i] -= a[u]
            return False

        return dfs(0)
```

**Độ phức tạp:** Time tệ nhất `O(4^n)` (cắt tỉa giúp rất nhiều trong thực tế), Space `O(n)` cho ngăn xếp đệ quy.

### Cách 3 — DP bitmask (`dp[mask]` = độ dài cạnh đang dựng dở)

**Ý tưởng:** `mask` là tập que đã dùng. Với một tập `mask` có thể sắp xếp hợp lệ, các cạnh trước đã đầy đúng `x`, nên cạnh đang dở có độ dài `tổng(mask) % x`. Ta chỉ cần biết **tập này có "đạt được" không** (có cách xếp mà không cạnh nào vượt `x`).

**Từng đoạn:**

1. `dp = [-1] * (1 << n)`; `dp[0] = 0` (chưa dùng que nào, cạnh dở dài `0`). `-1` nghĩa là "không đạt được".
2. Với mỗi `mask` mà `dp[mask] != -1`: thử thêm que `i` chưa dùng; điều kiện `dp[mask] + a[i] <= x` (cạnh dở cộng que không vượt `x`).
3. Trạng thái mới: `dp[mask | 1 << i] = (dp[mask] + a[i]) % x`. Dùng `% x` vì khi cạnh vừa đầy (`== x`) thì cạnh mới bắt đầu từ `0`.
4. Đáp án: `dp[(1 << n) - 1] == 0`: dùng hết que và cạnh cuối vừa đầy.

Thứ tự duyệt `mask` tăng dần là đủ vì `mask | 1 << i > mask` (mask mới luôn lớn hơn).

In [10]:
def make_square_bitmask(ms):
    canh, du = divmod(sum(ms), 4)
    if du != 0 or canh < max(ms):
        return False
    n = len(ms)
    dp = [-1] * (1 << n)
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] == -1:
            continue
        for i in range(n):
            if mask >> i & 1:
                continue                                   # que i đã dùng
            if dp[mask] + ms[i] > canh:
                continue                                   # đặt vào sẽ vượt cạnh
            moi = mask | (1 << i)
            dp[moi] = (dp[mask] + ms[i]) % canh
    return dp[(1 << n) - 1] == 0

for ms, ans in examples:
    print(ms, "->", make_square_bitmask(ms), "(mong doi", ans, ")")

[1, 1, 2, 2, 2] -> True (mong doi True )
[3, 3, 3, 3, 4] -> False (mong doi False )


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def makesquare(self, matchsticks: List[int]) -> bool:
        canh, du = divmod(sum(matchsticks), 4)
        if du != 0 or canh < max(matchsticks):
            return False
        n = len(matchsticks)
        dp = [-1] * (1 << n)
        dp[0] = 0
        for mask in range(1 << n):
            if dp[mask] == -1:
                continue
            for i in range(n):
                if mask >> i & 1:
                    continue
                if dp[mask] + matchsticks[i] > canh:
                    continue
                dp[mask | (1 << i)] = (dp[mask] + matchsticks[i]) % canh
        return dp[(1 << n) - 1] == 0
```

**Độ phức tạp:** Time `O(2^n * n)`, Space `O(2^n)`. Với `n = 15`: `32768 * 15 ≈ 5 × 10^5` thao tác, nhanh và **không phụ thuộc** vào việc may rủi của cắt tỉa.

### Cách 4 — Quay lui theo từng CẠNH (lấp đầy một cạnh rồi sang cạnh kế)

**Ý tưởng khác hẳn Cách 2:** thay vì "đặt từng que vào 4 cạnh", ta **dựng lần lượt từng cạnh**: chọn một tập que có tổng đúng `x` cho cạnh 1, rồi tiếp tục cạnh 2, cạnh 3. Cạnh thứ 4 tự đúng (phần còn lại có tổng `4x - 3x = x`).

**Từng đoạn:**

1. `used[i]`: que `i` đã dùng cho cạnh nào đó chưa.
2. `dfs(k, start, cur)`: đang dựng cạnh `k` (đã xong `k` cạnh), duyệt que từ chỉ số `start` trở đi (để **không sinh các hoán vị trùng**: chỉ chọn theo thứ tự chỉ số tăng), `cur` là tổng cạnh đang dựng.
3. Nếu `k == 3` → `True` (3 cạnh đầy, cạnh cuối tự đủ). Nếu `cur == x` → bắt đầu cạnh mới `dfs(k + 1, 0, 0)`.
4. Với `i >= start` chưa dùng và `cur + a[i] <= x`: đánh dấu, đệ quy `dfs(k, i + 1, cur + a[i])`, hoàn tác.
5. Mẹo thêm: nếu `cur == 0` mà không có que nào đầu cạnh dẫn tới thành công thì **cả bài** thất bại (que lớn nhất chưa dùng buộc phải nằm ở cạnh này).
6. Nhớ các `mask` đã thất bại (`bad`) để không lặp lại, vì cùng một tập que đã dùng ở đầu cạnh mới thì kết quả như nhau.

In [11]:
def make_square_side(ms):
    canh, du = divmod(sum(ms), 4)
    if du != 0 or canh < max(ms):
        return False
    a = sorted(ms, reverse=True)
    n = len(a)
    used = [False] * n
    bad = set()                                  # các mask (đầu cạnh mới) đã biết là thất bại

    def mask_hien_tai():
        return sum(1 << i for i in range(n) if used[i])

    def dfs(k, start, cur):
        if k == 3:
            return True
        if cur == canh:
            m = mask_hien_tai()
            if m in bad:
                return False
            if dfs(k + 1, 0, 0):
                return True
            bad.add(m)
            return False
        for i in range(start, n):
            if used[i] or cur + a[i] > canh:
                continue
            used[i] = True
            if dfs(k, i + 1, cur + a[i]):
                return True
            used[i] = False
            if cur == 0:                         # que đầu cạnh không dùng được -> cả bài hỏng
                return False
        return False

    return dfs(0, 0, 0)

for ms, ans in examples:
    print(ms, "->", make_square_side(ms), "(mong doi", ans, ")")

[1, 1, 2, 2, 2] -> True (mong doi True )
[3, 3, 3, 3, 4] -> False (mong doi False )


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def makesquare(self, matchsticks: List[int]) -> bool:
        canh, du = divmod(sum(matchsticks), 4)
        if du != 0 or canh < max(matchsticks):
            return False
        a = sorted(matchsticks, reverse=True)
        n = len(a)
        used = [False] * n
        bad = set()

        def dfs(k, start, cur):
            if k == 3:
                return True
            if cur == canh:
                m = sum(1 << i for i in range(n) if used[i])
                if m in bad:
                    return False
                if dfs(k + 1, 0, 0):
                    return True
                bad.add(m)
                return False
            for i in range(start, n):
                if used[i] or cur + a[i] > canh:
                    continue
                used[i] = True
                if dfs(k, i + 1, cur + a[i]):
                    return True
                used[i] = False
                if cur == 0:
                    return False
            return False

        return dfs(0, 0, 0)
```

**Độ phức tạp:** Time `O(2^n * n)` (nhờ nhớ mask thất bại; mỗi mask đầu cạnh chỉ được mở rộng một lần), Space `O(2^n)` cho `bad` trong trường hợp xấu nhất và `O(n)` cho đệ quy.

## 8. TRACE từng bước

Ta theo dõi Cách 2 trên `[1,1,2,2,2]` (đã sắp giảm dần `[2,2,2,1,1]`, `x = 2`). Mỗi dòng cho biết độ sâu `u`, que đang đặt, cạnh thử, trạng thái `edges` sau khi đặt và quyết định.

| u | que | thử cạnh | edges sau khi đặt | quyết định |
|---|-----|----------|-------------------|------------|
| 0 | 2 | 0 | [2,0,0,0] | hợp lệ (`2 <= 2`) → đi sâu |
| 1 | 2 | 0 | [4,0,0,0] | vượt `x` → hoàn tác |
| 1 | 2 | 1 | [2,2,0,0] | hợp lệ → đi sâu |
| 2 | 2 | 0 | [4,2,0,0] | vượt → hoàn tác |
| 2 | 2 | 1 | [2,4,0,0] | vượt → hoàn tác |
| 2 | 2 | 2 | [2,2,2,0] | hợp lệ → đi sâu |
| 3 | 1 | 0,1,2 | | vượt hết (hoặc bị bỏ qua vì đối xứng) |
| 3 | 1 | 3 | [2,2,2,1] | hợp lệ → đi sâu |
| 4 | 1 | 3 | [2,2,2,2] | hợp lệ, `u == n` → `True` |

Hàm dưới in đúng quá trình này.

In [12]:
def trace_backtrack(ms):
    canh, du = divmod(sum(ms), 4)
    print(f"tong = {sum(ms)}, canh x = {canh}, du = {du}, que max = {max(ms)}")
    if du != 0 or canh < max(ms):
        print("loai ngay bang dieu kien can -> False")
        return False
    a = sorted(ms, reverse=True)
    n = len(a)
    print("que sau khi sap giam dan:", a)
    edges = [0, 0, 0, 0]

    def dfs(u):
        if u == n:
            print("  " * u + f"u={u}: da dat het que, edges={edges} -> True")
            return True
        for i in range(4):
            if i > 0 and edges[i - 1] == edges[i]:
                print("  " * u + f"u={u}: que {a[u]} bo qua canh {i} (doi xung voi canh {i-1}, edges={edges})")
                continue
            edges[i] += a[u]
            ok = edges[i] <= canh
            print("  " * u + f"u={u}: que {a[u]} -> canh {i}, edges={edges}, {'hop le' if ok else 'VUOT -> hoan tac'}")
            if ok and dfs(u + 1):
                return True
            edges[i] -= a[u]
        print("  " * u + f"u={u}: het cach, quay lui")
        return False

    return dfs(0)

print("KET QUA:", trace_backtrack([1, 1, 2, 2, 2]))

tong = 8, canh x = 2, du = 0, que max = 2
que sau khi sap giam dan: [2, 2, 2, 1, 1]
u=0: que 2 -> canh 0, edges=[2, 0, 0, 0], hop le
  u=1: que 2 -> canh 0, edges=[4, 0, 0, 0], VUOT -> hoan tac
  u=1: que 2 -> canh 1, edges=[2, 2, 0, 0], hop le
    u=2: que 2 -> canh 0, edges=[4, 2, 0, 0], VUOT -> hoan tac
    u=2: que 2 bo qua canh 1 (doi xung voi canh 0, edges=[2, 2, 0, 0])
    u=2: que 2 -> canh 2, edges=[2, 2, 2, 0], hop le
      u=3: que 1 -> canh 0, edges=[3, 2, 2, 0], VUOT -> hoan tac
      u=3: que 1 bo qua canh 1 (doi xung voi canh 0, edges=[2, 2, 2, 0])
      u=3: que 1 bo qua canh 2 (doi xung voi canh 1, edges=[2, 2, 2, 0])
      u=3: que 1 -> canh 3, edges=[2, 2, 2, 1], hop le
        u=4: que 1 -> canh 0, edges=[3, 2, 2, 1], VUOT -> hoan tac
        u=4: que 1 bo qua canh 1 (doi xung voi canh 0, edges=[2, 2, 2, 1])
        u=4: que 1 bo qua canh 2 (doi xung voi canh 1, edges=[2, 2, 2, 1])
        u=4: que 1 -> canh 3, edges=[2, 2, 2, 2], hop le
          u=5: da dat het qu

Bây giờ trace trường hợp `false` `[3,3,3,3,4]` để thấy quay lui thất bại thế nào (`x = 4`, que sau sắp xếp `[4,3,3,3,3]`).

In [13]:
print("KET QUA:", trace_backtrack([3, 3, 3, 3, 4]))

tong = 16, canh x = 4, du = 0, que max = 4
que sau khi sap giam dan: [4, 3, 3, 3, 3]
u=0: que 4 -> canh 0, edges=[4, 0, 0, 0], hop le
  u=1: que 3 -> canh 0, edges=[7, 0, 0, 0], VUOT -> hoan tac
  u=1: que 3 -> canh 1, edges=[4, 3, 0, 0], hop le
    u=2: que 3 -> canh 0, edges=[7, 3, 0, 0], VUOT -> hoan tac
    u=2: que 3 -> canh 1, edges=[4, 6, 0, 0], VUOT -> hoan tac
    u=2: que 3 -> canh 2, edges=[4, 3, 3, 0], hop le
      u=3: que 3 -> canh 0, edges=[7, 3, 3, 0], VUOT -> hoan tac
      u=3: que 3 -> canh 1, edges=[4, 6, 3, 0], VUOT -> hoan tac
      u=3: que 3 bo qua canh 2 (doi xung voi canh 1, edges=[4, 3, 3, 0])
      u=3: que 3 -> canh 3, edges=[4, 3, 3, 3], hop le
        u=4: que 3 -> canh 0, edges=[7, 3, 3, 3], VUOT -> hoan tac
        u=4: que 3 -> canh 1, edges=[4, 6, 3, 3], VUOT -> hoan tac
        u=4: que 3 bo qua canh 2 (doi xung voi canh 1, edges=[4, 3, 3, 3])
        u=4: que 3 bo qua canh 3 (doi xung voi canh 2, edges=[4, 3, 3, 3])
        u=4: het cach, quay lui
 

Và trace **DP bitmask** để thấy `dp[mask]` thay đổi ra sao trên `[1,1,2,2,2]` (chỉ in các mask đạt được, ghi bằng nhị phân; bit `i` ứng với que `i` theo thứ tự đề cho).

In [14]:
def trace_bitmask(ms):
    canh, du = divmod(sum(ms), 4)
    n = len(ms)
    print("canh =", canh, "| que =", ms)
    dp = [-1] * (1 << n)
    dp[0] = 0
    dat_duoc = 0
    for mask in range(1 << n):
        if dp[mask] == -1:
            continue
        dat_duoc += 1
        if dat_duoc <= 12 or mask == (1 << n) - 1:
            print(f"mask={mask:0{n}b} dp={dp[mask]}")
        for i in range(n):
            if mask >> i & 1 or dp[mask] + ms[i] > canh:
                continue
            dp[mask | (1 << i)] = (dp[mask] + ms[i]) % canh
    print("so mask dat duoc:", dat_duoc, "/", 1 << n)
    print("mask day du", format((1 << n) - 1, f"0{n}b"), "co dp =", dp[(1 << n) - 1], "-> ", dp[(1 << n) - 1] == 0)

trace_bitmask([1, 1, 2, 2, 2])

canh = 2 | que = [1, 1, 2, 2, 2]
mask=00000 dp=0
mask=00001 dp=1
mask=00010 dp=1
mask=00011 dp=0
mask=00100 dp=0
mask=00101 dp=1
mask=00110 dp=1
mask=00111 dp=0
mask=01000 dp=0
mask=01001 dp=1
mask=01010 dp=1
mask=01011 dp=0
mask=11111 dp=0
so mask dat duoc: 32 / 32
mask day du 11111 co dp = 0 ->  True


## 9. Code nộp LeetCode

Ta nộp **Cách 2** (quay lui có cắt tỉa, đặt que dài trước, bỏ qua cạnh đối xứng) vì ngắn gọn và đủ nhanh cho `n <= 15`. Cách 3 (bitmask) là phương án an toàn nhất về thời gian tệ nhất.

```python
class Solution:
    def makesquare(self, matchsticks: List[int]) -> bool:
        canh, du = divmod(sum(matchsticks), 4)
        if du != 0 or canh < max(matchsticks):
            return False
        a = sorted(matchsticks, reverse=True)
        n = len(a)
        edges = [0, 0, 0, 0]

        def dfs(u):
            if u == n:
                return True
            for i in range(4):
                if i > 0 and edges[i - 1] == edges[i]:
                    continue
                edges[i] += a[u]
                if edges[i] <= canh and dfs(u + 1):
                    return True
                edges[i] -= a[u]
            return False

        return dfs(0)
```

In [15]:
class Solution:
    def makesquare(self, matchsticks: List[int]) -> bool:
        canh, du = divmod(sum(matchsticks), 4)
        if du != 0 or canh < max(matchsticks):
            return False
        a = sorted(matchsticks, reverse=True)
        n = len(a)
        edges = [0, 0, 0, 0]

        def dfs(u):
            if u == n:
                return True
            for i in range(4):
                if i > 0 and edges[i - 1] == edges[i]:
                    continue
                edges[i] += a[u]
                if edges[i] <= canh and dfs(u + 1):
                    return True
                edges[i] -= a[u]
            return False

        return dfs(0)

print(Solution().makesquare([1, 1, 2, 2, 2]))   # True
print(Solution().makesquare([3, 3, 3, 3, 4]))   # False

True
False


## 10. Vì sao thuật toán đúng

**Tính đầy đủ (không bỏ sót).** Quay lui thử mọi cách gán que vào cạnh, trừ hai loại nhánh bị loại:

- Nhánh có `edges[i] > x`: về sau độ dài cạnh chỉ tăng, nên **không bao giờ** về đúng `x` → loại là an toàn.
- Nhánh bỏ qua cạnh `i` vì `edges[i-1] == edges[i]`: hai cạnh này **hoàn toàn giống nhau** ở thời điểm đó, nên mọi cách hoàn thành khi đặt vào `i` ứng với một cách hoàn thành khi đặt vào `i-1` (hoán đổi nhãn). Mất nhánh `i` không mất đáp án.

**Tính đúng khi kết luận `True`.** Khi `u == n`, mọi que đã đặt vào cạnh nào đó và không cạnh nào vượt `x`. Tổng bốn cạnh là `S = 4x`. Nếu một cạnh `< x` thì tổng ba cạnh còn lại `> 3x`, nhưng mỗi cạnh `<= x` cho tổng `<= 3x`, mâu thuẫn. Vậy cả bốn cạnh đều đúng bằng `x`.

**Điều kiện cắt sớm** (`S % 4 != 0`, que dài nhất `> x`) là điều kiện cần, nên trả `False` là đúng.

**DP bitmask:** `dp[mask] != -1` nghĩa là tồn tại thứ tự thêm que cho `mask` sao cho độ dài cạnh dở luôn `<= x`; khi đó độ dài dở bằng `tổng(mask) mod x` (vì các cạnh đã đầy là bội của `x`), nên `dp[mask]` được xác định duy nhất bởi `mask`: không cần lưu thêm thông tin nào khác. `dp[đủ] == 0` nghĩa là tổng là bội của `x` và không còn cạnh dở.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| 1. Vét cạn `4^n` | `O(4^n * n)` | `O(n)` | chỉ dùng làm oracle (`n <= 8`) |
| 2. Quay lui cắt tỉa (nộp) | tệ nhất `O(4^n)` | `O(n)` | rất nhanh thực tế nhờ cắt tỉa |
| 3. DP bitmask | `O(2^n * n)` | `O(2^n)` | thời gian ổn định, dễ chứng minh |
| 4. Quay lui theo cạnh + nhớ mask | `O(2^n * n)` | `O(2^n)` | bản chất là DP bitmask top-down |

Với `n <= 15`: Cách 3 cần khoảng `5 × 10^5` phép tính; đủ nhanh dù trong Python.

## 12. Bộ test

Ta dùng `make_square_brute` (Cách 1) làm **oracle độc lập** cho input nhỏ, rồi đối chiếu 3 cách còn lại + `Solution` bằng `assert`.

In [16]:
cac_cach = {
    "backtrack": make_square_backtrack,
    "bitmask":   make_square_bitmask,
    "side":      make_square_side,
    "Solution":  lambda ms: Solution().makesquare(list(ms)),
}

def so_het(ms, oracle=None):
    kq = {ten: f(list(ms)) for ten, f in cac_cach.items()}
    mong_doi = oracle if oracle is not None else make_square_brute(list(ms))
    for ten, v in kq.items():
        assert v == mong_doi, (ten, ms, v, mong_doi)
    return mong_doi

# Ví dụ của đề
assert so_het([1, 1, 2, 2, 2]) is True
assert so_het([3, 3, 3, 3, 4]) is False
print("Vi du de OK")

Vi du de OK


In [17]:
# Edge case
tests = [
    ([1], False),                        # chỉ 1 que
    ([1, 1, 1], False),                  # chưa đủ 4 que
    ([1, 1, 1, 1], True),                # 4 que bằng nhau
    ([2, 2, 2, 2, 2, 2, 2, 2], True),    # 8 que bằng nhau
    ([5, 5, 5, 5, 4], False),            # tổng 24: mỗi cạnh 6, que 5 và 4 không ghép được
    ([1, 2, 3, 4], False),               # tổng 10 không chia hết cho 4
    ([4, 4, 4, 4, 4], False),            # tổng 20 -> cạnh 5, que 4 không bù được
    ([10**8] * 4, True),                 # giá trị lớn
    ([1, 1, 1, 1, 2, 2, 2, 2], True),    # cạnh 3: 1+2 bốn lần
    ([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], True),   # 12 que bằng nhau
    ([1, 1, 2, 2, 2, 3, 3], False),      # tổng 14 lẻ với 4
    ([6, 6, 6, 6, 3, 3, 3, 3], True),    # cạnh 9: 6+3 bốn lần
]
for ms, mong_doi in tests:
    kq = so_het(ms, make_square_brute(ms) if len(ms) <= 8 else None)
    assert kq == mong_doi, (ms, kq, mong_doi)
print("Edge case OK:", len(tests), "test")

Edge case OK: 12 test


**Phản ví dụ** cho các heuristic sai thường gặp:

- `[5, 5, 5, 5, 4]`: tổng `24` chia hết cho 4 và que lớn nhất `5 <= 6` mà vẫn **không** xếp được. Vậy hai điều kiện cần **không đủ**.
- Tham lam "luôn đặt que vào cạnh đang ngắn nhất" có thể sai; mục 13 (Lỗi 3) cho một phản ví dụ tìm bằng máy.

In [18]:
# Đối chiếu ngẫu nhiên có seed: input nhỏ (n <= 8) đối chiếu với vét cạn
random.seed(473)
dem_true = 0
for _ in range(400):
    n = random.randint(1, 8)
    ms = [random.randint(1, 6) for _ in range(n)]
    dem_true += so_het(ms)
print("n <= 8: 400 test OK, so test True:", dem_true)

n <= 8: 400 test OK, so test True: 19


In [19]:
# Input sinh sẵn từ hình vuông thật (chắc chắn True), n tới 15: so 4 cách với nhau
def sinh_hinh_vuong(x, n, rng):
    """Chia 4 cạnh dài x thành tổng n que (n >= 4), rồi xáo trộn."""
    que_moi_canh = [1, 1, 1, 1]
    for _ in range(n - 4):
        que_moi_canh[rng.randrange(4)] += 1
    ms = []
    for k in que_moi_canh:
        # chia x thành k phần dương
        cat = sorted(rng.sample(range(1, x), k - 1)) if k > 1 else []
        moc = [0] + cat + [x]
        ms += [moc[i + 1] - moc[i] for i in range(k)]
    rng.shuffle(ms)
    return ms

rng = random.Random(4730)
for _ in range(150):
    n = rng.randint(4, 15)
    x = rng.randint(n, 40)
    ms = sinh_hinh_vuong(x, n, rng)
    assert sum(ms) == 4 * x and len(ms) == n
    kq = [f(list(ms)) for f in cac_cach.values()]
    assert all(kq), (ms, kq)
print("150 hinh vuong that (n <= 15): ca 4 cach deu True")

# Cũng input ngẫu nhiên không chắc True, so 3 cách nhanh với nhau (n tới 15)
rng = random.Random(4731)
dem = 0
t0 = time.perf_counter()
for _ in range(300):
    n = rng.randint(4, 15)
    ms = [rng.randint(1, 9) for _ in range(n)]
    s = sum(ms)
    if s % 4:
        ms[0] += 4 - s % 4                    # ép tổng chia hết cho 4 để test nhiều ca khó
    kq = {ten: f(list(ms)) for ten, f in cac_cach.items()}
    assert len(set(kq.values())) == 1, (ms, kq)
    dem += list(kq.values())[0]
print("300 test ngau nhien (n tới 15) nhat quan, so True:", dem, f"| mat {time.perf_counter() - t0:.2f}s")

150 hinh vuong that (n <= 15): ca 4 cach deu True


300 test ngau nhien (n tới 15) nhat quan, so True: 156 | mat 1.69s


In [20]:
# Hiệu năng: ca xấu cho quay lui (tổng đúng, nhiều que bằng nhau, nhưng không xếp được)
xau2 = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6]   # tổng 48, cạnh 12: 3*4 mỗi cạnh nên True
xau3 = [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4]   # tổng 73 lẻ
xau4 = [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2]   # tổng 58, du 2 -> loại ngay
xau5 = [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4]   # tổng 84, cạnh 21 là số lẻ mà mọi que đều chẵn -> không ghép được
for ms in (xau2, xau3, xau4, xau5):
    t0 = time.perf_counter()
    kq = {ten: f(list(ms)) for ten, f in cac_cach.items()}
    assert len(set(kq.values())) == 1
    print(f"{str(ms)[:42]:44} -> {list(kq.values())[0]!s:5} ({time.perf_counter() - t0:.3f}s)")

[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,   -> True  (0.042s)
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4,   -> False (0.000s)
[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,   -> False (0.000s)
[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4,   -> False (0.004s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra tổng chia hết cho 4 / que dài hơn cạnh

Nếu bỏ hai kiểm tra này, quay lui vẫn cho kết quả đúng (vì không thể đặt hết que khi `tổng > 4 * (tổng // 4)`), nhưng phải **đi vào cả một cây tìm kiếm vô ích** thay vì trả lời ngay trong `O(n)`. Ta đếm số lần gọi `dfs`.

In [21]:
def dem_dfs(ms, kiem_tra_som):
    canh, du = divmod(sum(ms), 4)
    if kiem_tra_som and (du != 0 or canh < max(ms)):
        return False, 0                      # trả lời ngay, 0 lần gọi dfs
    a = sorted(ms, reverse=True)
    edges = [0, 0, 0, 0]
    dem = [0]
    def dfs(u):
        dem[0] += 1
        if u == len(a):
            return True
        for i in range(4):
            if i and edges[i - 1] == edges[i]:
                continue
            edges[i] += a[u]
            if edges[i] <= canh and dfs(u + 1):
                return True
            edges[i] -= a[u]
        return False
    return dfs(0), dem[0]

ms = [3] * 14 + [1]                          # tổng 43, không chia hết cho 4
print("khong kiem tra som:", dem_dfs(ms, False))   # kết quả đúng nhưng tốn hàng nghìn lần gọi
print("co kiem tra som   :", dem_dfs(ms, True))

khong kiem tra som: (False, 2035)
co kiem tra som   : (False, 0)


**Code đúng:** `canh, du = divmod(sum(ms), 4); if du or canh < max(ms): return False`. Với `n = 15` còn chấp nhận được, nhưng trong bài lớn hơn (ví dụ họ hàng LeetCode 698) thiếu kiểm tra sớm có thể gây TLE.

### Lỗi 2 — Quên hoàn tác (`edges[i] -= que`)

Nếu quên hoàn tác khi nhánh hỏng, `edges` bị cộng dồn "rác" từ các nhánh trước và những nhánh sau bị loại oan.

In [22]:
def sai_quen_hoan_tac(ms):
    canh, du = divmod(sum(ms), 4)
    if du or canh < max(ms):
        return False
    a = sorted(ms, reverse=True)
    edges = [0, 0, 0, 0]
    def dfs(u):
        if u == len(a):
            return True
        for i in range(4):
            edges[i] += a[u]
            if edges[i] <= canh and dfs(u + 1):
                return True
            # SAI: thiếu edges[i] -= a[u]
        return False
    return dfs(0)

rng = random.Random(2)
tim = None
for _ in range(20000):
    ms = [rng.randint(1, 4) for _ in range(rng.randint(4, 8))]
    if sai_quen_hoan_tac(ms) != make_square_brute(ms):
        tim = ms
        break
print("input:", tim, "| ban sai =", sai_quen_hoan_tac(tim), "| dung =", Solution().makesquare(tim))

input: [3, 2, 1, 2, 4, 4] | ban sai = False | dung = True


**Vì sao sai:** khi một nhánh thất bại mà không trừ lại, `edges[i]` giữ giá trị "ma", các nhánh sau thấy cạnh đã đầy hơn thực tế và bị loại oan, dẫn tới `False` sai.

**Code đúng:** luôn đi cặp `edges[i] += que` ... `edges[i] -= que`.

### Lỗi 3 — Tham lam "luôn đặt que vào cạnh đang ngắn nhất" (không quay lui)

Tham lam không thử lại khi hỏng. Ta để máy tìm một ví dụ mà tham lam cho kết quả khác vét cạn.

In [23]:
def tham_lam(ms):
    canh, du = divmod(sum(ms), 4)
    if du or canh < max(ms):
        return False
    edges = [0, 0, 0, 0]
    for que in sorted(ms, reverse=True):
        i = edges.index(min(edges))                  # luôn đặt vào cạnh ngắn nhất
        edges[i] += que
    return all(e == canh for e in edges)

rng = random.Random(1)
tim = None
for _ in range(20000):
    n = rng.randint(4, 8)
    ms = [rng.randint(1, 6) for _ in range(n)]
    if tham_lam(ms) != make_square_brute(ms):
        tim = ms
        break
print("phan vi du tham lam:", tim, "| tham lam =", tham_lam(tim), "| dung =", make_square_brute(tim))

phan vi du tham lam: [3, 6, 2, 2, 3, 6, 2] | tham lam = False | dung = True


**Vì sao sai:** đặt que vào cạnh ngắn nhất lúc này có thể phá hỏng cách ghép đúng về sau (ví dụ cần hai que `3` bù cho nhau trên cùng một cạnh). Bài toán này là dạng NP-đầy-đủ nên **không có** tham lam đúng; cần quay lui/DP.

### Lỗi 4 — Không sắp xếp giảm dần và không bỏ qua cạnh đối xứng (đúng nhưng chậm)

Hai mẹo này không ảnh hưởng tính đúng nhưng ảnh hưởng thời gian. Ta đếm số lần gọi `dfs` trong ba biến thể trên cùng một input `False` khó.

In [24]:
def dem_goi(ms, sap_xep, bo_doi_xung):
    canh, du = divmod(sum(ms), 4)
    a = sorted(ms, reverse=True) if sap_xep else list(ms)
    edges = [0, 0, 0, 0]
    dem = [0]
    def dfs(u):
        dem[0] += 1
        if u == len(a):
            return True
        for i in range(4):
            if bo_doi_xung and i > 0 and edges[i - 1] == edges[i]:
                continue
            edges[i] += a[u]
            if edges[i] <= canh and dfs(u + 1):
                return True
            edges[i] -= a[u]
        return False
    kq = dfs(0)
    return kq, dem[0]

ms = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 5]        # tổng 100, cạnh 25; thử xem có xếp được không
for sx, dx in ((False, False), (True, False), (False, True), (True, True)):
    kq, dem = dem_goi(ms, sx, dx)
    print(f"sap giam dan={sx!s:5} bo doi xung={dx!s:5} -> kq={kq}, so lan goi dfs = {dem}")

sap giam dan=False bo doi xung=False -> kq=True, so lan goi dfs = 712
sap giam dan=True  bo doi xung=False -> kq=True, so lan goi dfs = 15
sap giam dan=False bo doi xung=True  -> kq=True, so lan goi dfs = 281
sap giam dan=True  bo doi xung=True  -> kq=True, so lan goi dfs = 15


Bạn thấy số lần gọi giảm mạnh khi bật cả hai mẹo. Đây là lý do notebook nộp bản có cả hai.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia thành `k` nhóm có tổng bằng nhau", `n <= 15..20` → **bitmask DP** hoặc **quay lui + cắt tỉa**.
- **Công thức nhanh:** cạnh `x = S / k`; nếu `S % k != 0` hoặc `max > x` → `False`.
- **Trạng thái bitmask:** `dp[mask]` chỉ cần lưu "độ dài nhóm đang dở" `= tổng(mask) % x` vì các nhóm trước đã đầy chẵn.
- **Cắt tỉa quay lui:** sắp xếp giảm dần; bỏ nhánh vượt; bỏ nhánh đối xứng (cạnh giống hệt nhau); bỏ nhánh khi que đầu cạnh thất bại.
- **Bài họ hàng:** LeetCode 698 *Partition to K Equal Sum Subsets* (tổng quát hóa `k`), 1723 *Find Minimum Time to Finish All Jobs*, 2305 *Fair Distribution of Cookies*.

## 15. Biến thể

1. **Tổng quát `k` cạnh** (LeetCode 698): thay `4` bằng `k`; code Cách 3 đổi `4` thành `k`, đáp án `dp[đủ] == 0`.
2. **Không bắt buộc dùng hết que**: chọn tập con que tạo được hình vuông có cạnh lớn nhất: thêm một vòng duyệt tập con (`3^n` nếu mỗi que thuộc 1 trong 4 cạnh hoặc bỏ).
3. **Chia thành 2 nhóm bằng nhau** (LeetCode 416 *Partition Equal Subset Sum*): `k = 2`, và vì `S` lớn có thể dùng DP tổng thay vì bitmask.
4. **Đếm số cách chia** thành 4 cạnh bằng nhau (không phân biệt cạnh): DP bitmask đếm thay vì kiểm tra tồn tại.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Vét cạn | gán que vào 4 cạnh, kiểm tra cuối | `O(4^n * n)` | `O(n)` | oracle cho `n <= 8`, học trực giác |
| 2. Quay lui cắt tỉa | đặt từng que, bỏ nhánh vượt/đối xứng | tệ nhất `O(4^n)` | `O(n)` | nộp bài, code ngắn, `n <= 15` |
| 3. DP bitmask | `dp[mask]` = độ dài cạnh dở | `O(2^n * n)` | `O(2^n)` | khi cần đảm bảo thời gian tệ nhất, `n <= 20` |
| 4. Quay lui theo cạnh + nhớ mask | dựng từng cạnh, nhớ mask thất bại | `O(2^n * n)` | `O(2^n)` | khi `k` cạnh lớn (bài 698) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `divmod(23, 4)`; (b) `bin((1 << 5) - 1)`; (c) `bool(0b10110 >> 2 & 1)`; (d) `Solution().makesquare([1,1,1,1])`; (e) `Solution().makesquare([5,5,5,5,4])`.

**Bài 2 (tính tay).** Với `[1,1,1,2,2,3]`: tính tổng, cạnh `x`, que lớn nhất và nói có xếp được không. Nếu được, hãy chỉ ra 4 cạnh.

**Bài 3 (điền TODO).** Hoàn thành `kiem_tra_dieu_kien(ms)` trả về `canh` (độ dài một cạnh) nếu qua mọi điều kiện cần, ngược lại trả về `-1`.

In [25]:
def kiem_tra_dieu_kien(ms):
    # TODO 1: tính canh, du = divmod(sum(ms), 4)
    # TODO 2: nếu du != 0 hoặc canh < max(ms) thì trả về -1
    # TODO 3: ngược lại trả về canh
    return None

# KHÔNG gọi kiem_tra_dieu_kien ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `dp_bitmask_todo(ms)` theo Cách 3 **mà không nhìn lại** mục 7 và kiểm tra bằng 4 ví dụ ở bài 1, 2.

**Bài 5 (biến thể / mini challenge).** Viết `chia_k(ms, k)` trả về `True` nếu chia được `ms` thành `k` nhóm có tổng bằng nhau (LeetCode 698). Kiểm tra: `chia_k([4,3,2,3,5,2,1], 4) == True`, `chia_k([1,2,3,4], 3) == False`. Gợi ý: Cách 3 với `4` đổi thành `k`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `(5, 3)`; (b) `'0b11111'`; (c) `10110 >> 2 = 101`, bit cuối là `1` → `True`; (d) `True`; (e) `False`. Bài 3: chỉ ba dòng đầu của Cách 2.
- **Mức 2 (vừa):** Bài 2: tổng `10`, dư `2` khi chia 4 → **không** xếp được. Bài 4: nhớ `dp[0] = 0` và `% canh` khi cập nhật.
- **Mức 3 (sâu):** Bài 5: thay `4` bằng `k`; cạnh `x = S / k`. Vì đề 698 có `n <= 16` nên bitmask `2^16 * 16 ≈ 10^6` vẫn chạy được.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
assert divmod(23, 4) == (5, 3)
assert bin((1 << 5) - 1) == "0b11111"
assert bool(0b10110 >> 2 & 1) is True
assert Solution().makesquare([1, 1, 1, 1]) is True
assert Solution().makesquare([5, 5, 5, 5, 4]) is False
print("Bài 1 OK")

# Bài 2
ms = [1, 1, 1, 2, 2, 3]
canh, du = divmod(sum(ms), 4)
print("Bài 2: tong =", sum(ms), "canh =", canh, "du =", du, "que max =", max(ms))
assert sum(ms) == 10 and du == 2
assert Solution().makesquare(ms) is False
print("Bài 2 OK")

# Bài 3
def kiem_tra_dieu_kien_ans(ms):
    canh, du = divmod(sum(ms), 4)
    if du != 0 or canh < max(ms):
        return -1
    return canh

assert kiem_tra_dieu_kien_ans([1, 1, 2, 2, 2]) == 2
assert kiem_tra_dieu_kien_ans([1, 1, 1, 2, 2, 3]) == -1
assert kiem_tra_dieu_kien_ans([10, 1, 1, 1, 1]) == -1       # tổng 14, dư 2
assert kiem_tra_dieu_kien_ans([8, 1, 1, 1, 1]) == -1        # tổng 12, cạnh 3 < que 8
print("Bài 3 OK")

# Bài 4
def dp_bitmask_todo_ans(ms):
    canh, du = divmod(sum(ms), 4)
    if du or canh < max(ms):
        return False
    n = len(ms)
    dp = [-1] * (1 << n)
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        for i in range(n):
            if not mask >> i & 1 and dp[mask] + ms[i] <= canh:
                dp[mask | 1 << i] = (dp[mask] + ms[i]) % canh
    return dp[-1] == 0

for ms, ans in [([1, 1, 2, 2, 2], True), ([3, 3, 3, 3, 4], False), ([1, 1, 1, 2, 2, 3], False), ([1, 1, 1, 1], True)]:
    assert dp_bitmask_todo_ans(ms) == ans, ms
print("Bài 4 OK")

# Bài 5
def chia_k(ms, k):
    S = sum(ms)
    if S % k:
        return False
    x = S // k
    if max(ms) > x:
        return False
    n = len(ms)
    dp = [-1] * (1 << n)
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        for i in range(n):
            if not mask >> i & 1 and dp[mask] + ms[i] <= x:
                dp[mask | 1 << i] = (dp[mask] + ms[i]) % x
    return dp[-1] == 0

assert chia_k([4, 3, 2, 3, 5, 2, 1], 4) is True
assert chia_k([1, 2, 3, 4], 3) is False
assert chia_k([1, 1, 2, 2, 2], 4) == Solution().makesquare([1, 1, 2, 2, 2])
random.seed(55)
for _ in range(200):
    ms = [random.randint(1, 5) for _ in range(random.randint(1, 8))]
    assert chia_k(ms, 4) == make_square_brute(ms), ms
print("Bài 5 OK")

Bài 1 OK
Bài 2: tong = 10 canh = 2 du = 2 que max = 3
Bài 2 OK
Bài 3 OK
Bài 4 OK


Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (quay lui đặt từng que vào 4 cạnh):**

```python
canh, du = divmod(sum(ms), 4)
if du or canh < max(ms):
    return False
a = sorted(ms, reverse=True)
edges = [0] * 4
def dfs(u):
    if u == len(a):
        return True
    for i in range(4):
        if i and edges[i - 1] == edges[i]:      # cạnh đối xứng
            continue
        edges[i] += a[u]
        if edges[i] <= canh and dfs(u + 1):
            return True
        edges[i] -= a[u]                        # hoàn tác
    return False
return dfs(0)
```

**Template DP bitmask:**

```python
dp = [-1] * (1 << n); dp[0] = 0
for mask in range(1 << n):
    if dp[mask] < 0: continue
    for i in range(n):
        if not mask >> i & 1 and dp[mask] + a[i] <= canh:
            dp[mask | 1 << i] = (dp[mask] + a[i]) % canh
return dp[-1] == 0
```

**Checklist:**

- [ ] Kiểm tra `tổng % 4 == 0` và `max <= tổng / 4` **trước** khi tìm kiếm.
- [ ] Sắp xếp **giảm dần** để que dài được đặt trước.
- [ ] Bỏ qua cạnh `i` khi `edges[i-1] == edges[i]` (tránh nhánh đối xứng).
- [ ] Luôn **hoàn tác** `edges[i] -= que` khi nhánh thất bại.
- [ ] Điều kiện kết thúc `u == n`: mọi que đã đặt mà không cạnh nào vượt.
- [ ] Thử tay `[1,1,2,2,2]` (True), `[3,3,3,3,4]` (False), `[5,5,5,5,4]` (False dù tổng chia hết), `[1,1,1]` (False).
- [ ] Đúng signature `class Solution` / `makesquare(self, matchsticks: List[int]) -> bool`; độ phức tạp `O(4^n)` thực tế rất nhanh, hoặc `O(2^n * n)` với bitmask.